# 01 - OCR-supported baseline

Evaluate images → untouched VLM transcript → frozen text formatting model → task JSON on the fixed eight validation sequences.

The default local models are `C:\MODELS\HF\Qwen2.5-VL-3B-Instruct` for image transcription and `C:\MODELS\HF\Qwen2.5-1.5B-Instruct` for JSON formatting. Override them with `MANGA_VLM_MODEL_PATH` and `MANGA_FORMATTER_MODEL_PATH`, or edit the paths in the setup cell. Both models load local files only. The formatter runs on CPU by default; it never receives labels or images.

Run the cells in order from a fresh kernel. The VLM generates and saves all validation transcripts first; it is then unloaded before the frozen text formatter loads. Raw transcripts are kept in the evaluation output folder under `raw_transcripts/predictions.jsonl`.

To retry formatting in a fresh kernel without regenerating images, run only the setup and dataset cells, then the **Stage 2** cells below. Do not run the VLM loading or training cells for that retry.


In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

ROOT = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / "Scripts" / "dataUtils.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate the repository root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from Scripts import dataUtils as dU
from Scripts import trainValSplit
from Scripts.evaluator import calculate_metrics, evaluate_model, inspect_prediction, load_prediction_records, display_metric_summary, prediction_report

SEED = 42
MODEL_ROOT = Path(r"C:\MODELS\HF")
MODEL_PATH = Path(os.environ.get("MANGA_VLM_MODEL_PATH") or MODEL_ROOT / "Qwen2.5-VL-3B-Instruct").expanduser()
FORMATTER_MODEL_PATH = Path(os.environ.get("MANGA_FORMATTER_MODEL_PATH") or MODEL_ROOT / "Qwen2.5-1.5B-Instruct").expanduser()
for model_role, model_path, override_name in (
    ("VLM", MODEL_PATH, "MANGA_VLM_MODEL_PATH"),
    ("JSON formatter", FORMATTER_MODEL_PATH, "MANGA_FORMATTER_MODEL_PATH"),
): 
    if not model_path.is_dir():
        raise FileNotFoundError(f"{model_role} model folder not found: {model_path}. Download the model there or set {override_name} to its local folder.")
    print(f"{model_role} model path: {model_path}")
LABELS_PATH = ROOT / "TaskData" / "development" / "labels.jsonl"
IMAGE_ROOT = ROOT / "TaskData" / "development" / "images"
SPLIT_PATH = ROOT / "outputs" / "canonical_split_seed42.json"
OUTPUT_DIR = ROOT / "outputs" / "ocr_base_baseline"
MAX_NEW_TOKENS = 1024
MINIMUM_SIMILARITY = 0.50
REPETITION_PENALTY = 1.05

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"CUDA available: {torch.cuda.is_available()}")

from Scripts.modelFormatter import (FormatterSequenceDataset, FormatterPromptBuilder, ModelJSONFormatter, export_transcript_dataset, format_transcript, format_saved_transcripts, transcript_to_pages)
WORKFLOW_DIR = ROOT / "notebooks-OCRsupported"
if str(WORKFLOW_DIR) not in sys.path:
    sys.path.insert(0, str(WORKFLOW_DIR))
from ocr_support import OCRPromptBuilder, prepare_ocr
OCR_LANGUAGES = ["en"]
OCR_CACHE_DIR = WORKFLOW_DIR / "datasets" / "ocr_cache"

TRANSCRIPT_PROMPT = (WORKFLOW_DIR / "prompts" / "transcript.txt").read_text(encoding="utf-8").strip()
FORMATTER_PROMPT = (WORKFLOW_DIR / "prompts" / "formatter.txt").read_text(encoding="utf-8").strip()
FORMATTER_DEVICE = os.environ.get("MANGA_FORMATTER_DEVICE", "cpu")
FORMATTER_MAX_NEW_TOKENS = 2048

RAW_TRANSCRIPT_DIR = OUTPUT_DIR / "raw_transcripts"
TRANSCRIPT_PATH = RAW_TRANSCRIPT_DIR / "predictions.jsonl"


In [ ]:
labels = dU.load_labels(LABELS_PATH)
all_ids = list(labels)
train_ids, validation_ids = trainValSplit.fit_seq(all_ids, v_size=0.10, random_state=SEED)
split = {"seed": SEED, "validation_fraction": 0.10, "train_ids": train_ids, "validation_ids": validation_ids}
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
if SPLIT_PATH.is_file():
    existing_split = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
    if existing_split != split:
        raise RuntimeError(f"Existing split differs from the seed-42 split: {SPLIT_PATH}")
else:
    SPLIT_PATH.write_text(json.dumps(split, indent=2), encoding="utf-8")

validation_dataset = FormatterSequenceDataset(LABELS_PATH, IMAGE_ROOT, validation_ids)
assert len(all_ids) == 80
assert len(train_ids) == 72 and len(validation_dataset) == 8
assert set(train_ids).isdisjoint(validation_ids)
display(pd.Series({"all sequences": len(all_ids), "training": len(train_ids), "validation": len(validation_dataset)}, name="count").to_frame())
display(pd.DataFrame({"validation_sequence_id": validation_ids}))

train_dataset = FormatterSequenceDataset(LABELS_PATH, IMAGE_ROOT, train_ids)
export_transcript_dataset(train_dataset, WORKFLOW_DIR / "datasets" / "train.jsonl")
export_transcript_dataset(validation_dataset, WORKFLOW_DIR / "datasets" / "validation.jsonl")
assert all(sample["transcript_target"] for sample in train_dataset)
print(train_dataset[0]["transcript_target"])

workflow_split_path = WORKFLOW_DIR / "datasets" / "split_seed42.json"
workflow_split_path.parent.mkdir(parents=True, exist_ok=True)
workflow_split_path.write_text(json.dumps(split, indent=2), encoding="utf-8")


In [ ]:
# OCR sees only source images, never reference text or speaker labels.
# Run before loading the VLM; EasyOCR uses CPU and cached full-resolution pages.
ocr_hints = prepare_ocr(validation_dataset, OCR_CACHE_DIR, OCR_LANGUAGES)
sample = validation_dataset[0]
for page, image_path in enumerate(sample["image_paths"], 1):
    print(f"OCR PAGE {page}: {image_path.name}")
    display(pd.DataFrame(ocr_hints[str(image_path.resolve())], columns=["box", "text", "confidence"]))


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True, min_pixels=128 * 28 * 28, max_pixels=256 * 28 * 28)
prompt_builder = OCRPromptBuilder(TRANSCRIPT_PROMPT, ocr_hints)
eval_collator = dU.MangaCollator(processor, process_vision_info, prompt_builder, train_mode=False, return_metadata=True)
sample_inputs, sample_metadata = eval_collator([validation_dataset[0]])
assert "labels" not in sample_inputs
assert sample_metadata[0]["target"] == validation_dataset[0]["target"]
print(f"Prompt tokens for first validation sequence: {sample_inputs['input_ids'].shape[1]}")
del sample_inputs, sample_metadata



experiment = {"pipeline": "images + OCR hints -> VLM transcript -> text model -> JSON", "ocr": {"engine": "easyocr", "languages": OCR_LANGUAGES, "gpu": False, "detail": 1, "paragraph": False, "cache_version": 1}, "vlm_model": str(MODEL_PATH), "formatter_model": str(FORMATTER_MODEL_PATH), "formatter_device": FORMATTER_DEVICE, "transcript_prompt": TRANSCRIPT_PROMPT, "formatter_prompt": FORMATTER_PROMPT, "split": split, "max_new_tokens": MAX_NEW_TOKENS, "formatter_max_new_tokens": FORMATTER_MAX_NEW_TOKENS, "repetition_penalty": REPETITION_PENALTY, "min_pixels": 128 * 28 * 28, "max_pixels": 256 * 28 * 28}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "experiment.json").write_text(json.dumps(experiment, indent=2), encoding="utf-8")


In [ ]:
# Release a formatter left from an earlier run before loading the VLM.
globals().pop("formatter", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the 4-bit VLM baseline")
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_PATH, local_files_only=True, torch_dtype=torch.bfloat16, quantization_config=quantization_config, device_map={"": 0})
base_model.eval()
print(f"Loaded base model on {next(base_model.parameters()).device}")

## Preview a raw VLM transcript

This stage uses only the VLM. JSON formatting and scoring happen after all eight transcripts have been saved and the VLM has been unloaded. Change `SAMPLE_INDEX` to inspect another validation sample.

Speaker IDs must be generated by the VLM from the images. The preview validates the required page headers and speaker/text lines and displays the extracted character IDs. Invalid transcripts remain baseline failures; the text formatter cannot reconstruct missing speaker identity.


In [ ]:
SAMPLE_INDEX = 0
sample = validation_dataset[SAMPLE_INDEX]
sample_inputs, sample_metadata = eval_collator([sample])
device = next(base_model.parameters()).device
sample_inputs = {name: value.to(device) if torch.is_tensor(value) else value for name, value in sample_inputs.items()}

with torch.inference_mode():
    generated = base_model.generate(
        **sample_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        repetition_penalty=REPETITION_PENALTY,
    )

prompt_width = sample_inputs["input_ids"].shape[1]
prediction = processor.decode(generated[0, prompt_width:], skip_special_tokens=True, clean_up_tokenization_spaces=False)
raw_transcript = prediction
print("RAW VLM TRANSCRIPT:\n", raw_transcript)
preview_generated_tokens = generated.shape[1] - prompt_width
print(f"Generated tokens: {preview_generated_tokens}/{MAX_NEW_TOKENS}")
if preview_generated_tokens >= MAX_NEW_TOKENS:
    print("WARNING: Token limit reached; the transcript may be incomplete or stuck repeating.")

# Inspect only the generated transcript; never fill missing speakers from targets.
try:
    preview_pages = transcript_to_pages(raw_transcript)
except ValueError as error:
    print(f"INVALID VLM TRANSCRIPT: {error}")
    print("Expected PAGE 1/2/3 headers and speaker<TAB>text for every item.")
    print("Missing speaker IDs must be generated from the images; Stage 2 cannot infer them.")
else:
    preview_items = [
        {"page": page_number, **item}
        for page_number, page in enumerate(preview_pages, 1)
        for item in page
    ]
    character_ids = sorted(
        {item["speaker"] for item in preview_items if item["speaker"].startswith("character")},
        key=lambda speaker: int(speaker.removeprefix("character")),
    )
    print("VALID TRANSCRIPT FORMAT (speaker attribution still requires visual review)")
    print("Identified characters:", ", ".join(character_ids) or "none")
    print("UNKNOWN items:", sum(item["speaker"] == "UNKNOWN" for item in preview_items))
    display(pd.DataFrame(preview_items, columns=["page", "speaker", "text"]))

del sample_inputs, sample_metadata, generated
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Stage 1: generate every VLM transcript without loading the formatter.
evaluate_model(
    model=base_model,
    processor=processor,
    dataset=validation_dataset,
    collator=eval_collator,
    batch_size=1,
    max_new_tokens=MAX_NEW_TOKENS,
    minimum_similarity=MINIMUM_SIMILARITY,
    repetition_penalty=REPETITION_PENALTY,
    output_dir=RAW_TRANSCRIPT_DIR,
    generation_only=True,
)
print(f"Saved raw VLM transcripts to {TRANSCRIPT_PATH}")


In [ ]:
import json
from pathlib import Path
from pprint import pprint

path = Path("outputs/ocr_base_baseline/raw_transcripts/predictions.jsonl")
if not path.exists():  # When the notebook runs from a subfolder
    path = Path("..") / path

records = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print(f"Records: {len(records)}")
print("Available keys:", sorted({key for record in records for key in record}))

ROW = 2
KEY = "prediction"  # e.g. target, sequence_id, image_input_diagnostics

value = records[ROW][KEY]
seq = records[ROW]['sequence_id']
print(seq)
print(f"\nRecord {ROW} | {KEY}\n")
if isinstance(value, str):
    print(value)
else:
    pprint(value, sort_dicts=False)

## Stage 2: unload the VLM and format saved transcripts

This cell reads the saved transcripts from disk. The formatter receives only transcript text; targets and images remain evaluation metadata. Training state is also released in the LoRA notebook. Formatting failures remain in the scored outputs.


In [ ]:
# Confirm all holdout transcripts are saved before releasing the VLM.
saved_transcripts = load_prediction_records(TRANSCRIPT_PATH)
if sorted(row["sequence_id"] for row in saved_transcripts) != sorted(validation_ids):
    raise RuntimeError("Saved transcripts do not match the complete validation split; rerun Stage 1")
del saved_transcripts

# Drop every notebook reference to the VLM and training state before formatting.
for name in ("trainer", "model", "base_model", "processor", "train_collator",
             "eval_collator", "sample_inputs", "sample_metadata", "generated",
             "formatter"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("VLM and training state released; loading the text formatter next.")

formatter = ModelJSONFormatter(
    FORMATTER_MODEL_PATH, FORMATTER_PROMPT,
    device=FORMATTER_DEVICE, max_new_tokens=FORMATTER_MAX_NEW_TOKENS,
)
try:
    baseline_metrics = format_saved_transcripts(
        TRANSCRIPT_PATH, formatter, OUTPUT_DIR,
        minimum_similarity=MINIMUM_SIMILARITY,
    )
finally:
    del formatter
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print(f"Saved formatted JSON predictions and metrics to {OUTPUT_DIR}")


Compare the model formatter with a tolerant deterministic parser on the same transcripts. The model remains the main pipeline; parser predictions and metrics are saved separately.


In [ ]:
# Compare serialization on the same saved VLM transcripts; no model reload.
parser_metrics = format_saved_transcripts(
    TRANSCRIPT_PATH, lambda text: format_transcript(text, tolerant=True),
    OUTPUT_DIR / "tolerant_parser", minimum_similarity=MINIMUM_SIMILARITY,
)
display(pd.DataFrame([
    {"formatter": "text model", "schema_compliance_rate": baseline_metrics["schema_compliance_rate"]},
    {"formatter": "tolerant parser", "schema_compliance_rate": parser_metrics["schema_compliance_rate"]},
]))
# Full task metrics and per-sequence failures are saved separately under tolerant_parser.


In [ ]:
SAMPLE_INDEX = globals().get("SAMPLE_INDEX", 0)
single_record = load_prediction_records(OUTPUT_DIR / "predictions.jsonl")[SAMPLE_INDEX]
print(f"SEQUENCE: {single_record['sequence_id']}")
print("RAW VLM TRANSCRIPT:\n", single_record["raw_transcript"])
print("RAW FORMATTER RESPONSE:\n", single_record["raw_formatter_prediction"])
print("FORMATTER ERROR:", single_record["formatter_error"])
print("TARGET:\n", json.dumps(single_record["target"], ensure_ascii=False, indent=2))
single_metrics = calculate_metrics([single_record], minimum_similarity=MINIMUM_SIMILARITY)
single_report = prediction_report(single_record, minimum_similarity=MINIMUM_SIMILARITY)
display_metric_summary(single_metrics, report=single_report)


In [ ]:
display_metric_summary(baseline_metrics)

records = load_prediction_records(OUTPUT_DIR / "predictions.jsonl")
per_sequence = []
for record in records:
    metrics = calculate_metrics([record], minimum_similarity=MINIMUM_SIMILARITY)
    per_sequence.append({"sequence_id": record["sequence_id"], "format_valid": bool(metrics["schema_compliance_rate"]), "text_error_rate": metrics["cer_valid_outputs"], "failure_penalty_cer": metrics["cer_including_failures_as_deletions"], "missing_dialogues": metrics["support"]["missing_dialogues"] if metrics["cer_evaluated_sequences"] else None, "extra_dialogues": metrics["support"]["extra_dialogues"] if metrics["cer_evaluated_sequences"] else None, "speaker_accuracy": metrics["matched_speaker_accuracy"], "dialogue_f1": metrics["dialogue_text_match_f1"], "exact_match": metrics["sequence_exact_match"], "generated_tokens": record["generated_tokens"], "hit_max_new_tokens": record["hit_max_new_tokens"], "repetition_loop": record["suspected_repetition_loop"]})
review = pd.DataFrame(per_sequence).sort_values(["format_valid", "failure_penalty_cer"], ascending=[True, False]).reset_index(drop=True)
display(review.style.format(na_rep="N/A"))
_ = inspect_prediction(OUTPUT_DIR / "predictions.jsonl", sequence_id=review.iloc[0]["sequence_id"], minimum_similarity=MINIMUM_SIMILARITY)

In [ ]:
records = load_prediction_records(OUTPUT_DIR / "predictions.jsonl")

for record in records:
    print(
        record["sequence_id"],
        "tokens:", record["generated_tokens"],
        "hit limit:", record["hit_max_new_tokens"],
        "EOS:", record["reached_eos"],
    )

In [ ]:
import json
from pathlib import Path
from pprint import pprint

path = Path("outputs/ocr_base_baseline/predictions.jsonl")
if not path.exists():  # When the notebook runs from a subfolder
    path = Path("..") / path

records = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print(f"Records: {len(records)}")
print("Available keys:", sorted({key for record in records for key in record}))

ROW = 1
KEY = "formatter_error"  # e.g. target, sequence_id, image_input_diagnostics

value = records[ROW][KEY]
seq = records[ROW]['sequence_id']
print(seq)
print(f"\nRecord {ROW} | {KEY}\n")
if isinstance(value, str):
    print(value)
else:
    pprint(value, sort_dicts=False)

## Official task score: primary experiment benchmark

Use the unmodified `TaskData/score.py` to judge text order and balanced text/speaker quality on the fixed holdout. Inspect invalid outputs and missing sequences before accepting an improvement.


In [ ]:
from Scripts.taskScore import score_saved_predictions

official_score = score_saved_predictions(OUTPUT_DIR / "predictions.jsonl", LABELS_PATH, validation_ids, OUTPUT_DIR / "task_score")
display(pd.Series(official_score["macro"], name="official_base_score").to_frame())
print("Missing or invalid sequences:", official_score["missing_sequences"])
